# Homework 1 — Agentic RAG

RAG over the course lessons, then the same pipeline as an agent. Lessons pulled
from the repo at commit `8c1834d`, model `gpt-5.4-mini`.

In [1]:
from dotenv import load_dotenv

load_dotenv()

True

## Load the lessons

In [2]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]

## Q1 — Lesson pages

In [3]:
len(documents)

72

## Q2 — Search

`content` as text, `filename` as keyword.

In [4]:
from minsearch import Index

index = Index(text_fields=["content"], keyword_fields=["filename"])
index.fit(documents)

QUERY = "How does the agentic loop keep calling the model until it stops?"
index.search(QUERY, num_results=5)[0]["filename"]

'01-agentic-rag/lessons/14-agentic-loop.md'

## Q3 — RAG

`RAGBase` is built for the FAQ schema, so I override the schema-specific parts
and make `rag` return the token usage too.

In [5]:
from openai import OpenAI
from rag_helper import RAGBase


class RAG(RAGBase):
    def search(self, query, num_results=5):
        return self.index.search(query, num_results=num_results)

    def build_context(self, search_results):
        lines = []
        for doc in search_results:
            lines.append(doc["filename"])
            lines.append(doc["content"])
            lines.append("")
        return "\n".join(lines).strip()

    def llm(self, prompt):
        input_messages = [
            {"role": "developer", "content": self.instructions},
            {"role": "user", "content": prompt},
        ]
        return self.llm_client.responses.create(
            model=self.model, input=input_messages
        )

    def rag(self, query):
        search_results = self.search(query)
        prompt = self.build_prompt(query, search_results)
        response = self.llm(prompt)
        return response.output_text, response.usage


client = OpenAI()
rag = RAG(index=index, llm_client=client, model="gpt-5.4-mini")

answer, usage = rag.rag(QUERY)
usage.input_tokens

7111

## Q4 — Chunking

Sliding window, size 2000, step 1000.

In [6]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)
len(chunks)

295

## Q5 — RAG over chunks

In [7]:
chunk_index = Index(text_fields=["content"], keyword_fields=["filename"])
chunk_index.fit(chunks)

rag_chunks = RAG(index=chunk_index, llm_client=client, model="gpt-5.4-mini")
_, usage_c = rag_chunks.rag(QUERY)

usage.input_tokens, usage_c.input_tokens, round(usage.input_tokens / usage_c.input_tokens, 1)

(7111, 2294, 3.1)

## Q6 — Agent

Hand `search` to a toyaikit agent and let it decide when to call it. The counter
tracks how many times it does.

In [8]:
from toyaikit.tools import Tools
from toyaikit.llm import OpenAIClient
from toyaikit.chat.runners import OpenAIResponsesRunner

search_calls = 0


def search(query: str) -> list[dict]:
    """Search the course lessons for chunks matching the given query."""
    global search_calls
    search_calls += 1
    return chunk_index.search(query, num_results=5)


INSTRUCTIONS = (
    "You're a course teaching assistant. Answer the student's question using the "
    "search tool. Make multiple searches with different keywords before answering."
)

agent_tools = Tools()
agent_tools.add_tool(search)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=INSTRUCTIONS,
    llm_client=OpenAIClient(model="gpt-5.4-mini"),
)

runner.loop(prompt="How does the agentic loop work, and how is it different from plain RAG?")
search_calls

3

## Answers

| Q | Answer |
|---|--------|
| Q1 | 72 |
| Q2 | `01-agentic-rag/lessons/14-agentic-loop.md` |
| Q3 | ~7000 |
| Q4 | 295 |
| Q5 | 3x fewer |
| Q6 | 4 |